# 机器人 RL 实验室：MuJoCo + PPO 连续动作

逐格运行：安装依赖 → 训练 → 独立评估 → 导出并验证 ONNX。需要联网和支持执行 Python 的环境；Colab 可能需要登录。这里训练的是仿真倒立摆，不连接真实硬件。


In [ ]:
%pip install -q 'gymnasium[mujoco]==1.4.0' 'mujoco==4.0.1' 'stable-baselines3==2.9.2' 'onnx==1.21.0' 'onnxruntime==1.26.0'


## 1. 写入课程代码
代码随笔记本携带；不需要额外下载课程仓库。


In [ ]:
from pathlib import Path
Path('train_ppo.py').write_text('"""Train and independently evaluate a continuous-action MuJoCo PPO example.\n\nThis is a simulation example, not a robot driver.  It never connects to hardware.\n"""\nimport argparse\nimport json\nimport time\nfrom pathlib import Path\n\nimport gymnasium as gym\nimport mujoco\nimport numpy as np\nimport stable_baselines3\nimport torch\nfrom stable_baselines3 import PPO\nfrom stable_baselines3.common.env_util import make_vec_env\n\n\nENV_ID = "InvertedPendulum-v5"\n\n\ndef evaluate(model, seeds):\n    """Evaluate independent episodes. The random baseline uses identical reset seeds."""\n    env = gym.make(ENV_ID)\n    scores, lengths, endings = [], [], []\n    for seed in seeds:\n        observation, _ = env.reset(seed=int(seed))\n        rng = np.random.default_rng(int(seed) + 100_000)\n        total, length = 0.0, 0\n        while True:\n            if model is None:\n                action = rng.uniform(env.action_space.low, env.action_space.high).astype(np.float32)\n            else:\n                action, _ = model.predict(observation, deterministic=True)\n            observation, reward, terminated, truncated, _ = env.step(action)\n            total += float(reward)\n            length += 1\n            if terminated or truncated:\n                scores.append(total)\n                lengths.append(length)\n                endings.append("terminated" if terminated else "time_limit")\n                break\n    env.close()\n    return {\n        "mean_return": round(float(np.mean(scores)), 3),\n        "min_return": round(float(np.min(scores)), 3),\n        "max_return": round(float(np.max(scores)), 3),\n        "mean_steps": round(float(np.mean(lengths)), 3),\n        "episodes_reaching_time_limit": endings.count("time_limit"),\n        "returns": scores,\n        "lengths": lengths,\n        "endings": endings,\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="MuJoCo + PPO simulation-only learning example")\n    parser.add_argument("--steps", type=int, default=60_000, help="training interaction budget")\n    parser.add_argument("--episodes", type=int, default=20, help="independent evaluation episodes")\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--out", type=Path, default=Path("runs/pendulum"))\n    args = parser.parse_args()\n    if args.steps < 1 or args.episodes < 1:\n        parser.error("--steps and --episodes must be positive")\n    args.out.mkdir(parents=True, exist_ok=True)\n    torch.set_num_threads(1)\n\n    # These seeds are separate from the training seed. The two agents see the\n    # same initial conditions; random actions have their own reproducible RNG.\n    test_seeds = list(range(100_000, 100_000 + args.episodes))\n    print("Evaluating random baseline...", flush=True)\n    baseline = evaluate(None, test_seeds)\n    env = make_vec_env(ENV_ID, n_envs=4, seed=args.seed)\n    model = PPO("MlpPolicy", env, seed=args.seed, verbose=1, device="cpu", n_steps=512,\n                batch_size=128, n_epochs=10, learning_rate=3e-4)\n    started = time.perf_counter()\n    model.learn(total_timesteps=args.steps, progress_bar=False)\n    elapsed = time.perf_counter() - started\n    model_path = args.out / "ppo_pendulum.zip"\n    model.save(model_path)\n    env.close()\n\n    # Reload the artifact before evaluation; this verifies the saved policy path.\n    reloaded = PPO.load(model_path, device="cpu")\n    print("Evaluating reloaded trained policy...", flush=True)\n    trained = evaluate(reloaded, test_seeds)\n    with gym.make(ENV_ID) as info_env:\n        dt = float(info_env.unwrapped.dt)\n    report = {\n        "environment": ENV_ID,\n        "scope": "simulation only; no real robot or hardware validation",\n        "versions": {"gymnasium": gym.__version__, "mujoco": mujoco.__version__,\n                     "stable_baselines3": stable_baselines3.__version__, "torch": torch.__version__},\n        "training_seed": args.seed,\n        "requested_training_steps": args.steps,\n        "actual_training_steps": int(model.num_timesteps),\n        "training_seconds": round(elapsed, 2),\n        "evaluation_seeds": test_seeds,\n        "control_interval_seconds": dt,\n        "observation_order": ["cart_position_m", "pole_angle_rad", "cart_velocity_m_per_s", "pole_angular_velocity_rad_per_s"],\n        "action": {"meaning": "cart force", "unit": "N", "low": -3.0, "high": 3.0},\n        "random_baseline": baseline,\n        "trained_policy": trained,\n    }\n    (args.out / "evaluation.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)\n    print(f"Saved model and measured evaluation to {args.out}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
Path('export_onnx.py').write_text('"""Export a deterministic PPO policy and check ONNX against SB3 predictions.\n\nThe action clamp is included. The exported model is for the simulated example,\nnot firmware and not a driver for a physical robot.\n"""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport onnx\nimport onnxruntime as ort\nimport torch\nfrom stable_baselines3 import PPO\n\n\nclass ExportPolicy(torch.nn.Module):\n    def __init__(self, policy, low, high):\n        super().__init__()\n        self.policy = policy\n        self.register_buffer("low", torch.as_tensor(low, dtype=torch.float32))\n        self.register_buffer("high", torch.as_tensor(high, dtype=torch.float32))\n\n    def forward(self, observation):\n        action, _, _ = self.policy(observation, deterministic=True)\n        return torch.minimum(torch.maximum(action, self.low), self.high)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--model", type=Path, default=Path("runs/pendulum/ppo_pendulum.zip"))\n    parser.add_argument("--out", type=Path, default=Path("runs/pendulum/policy.onnx"))\n    args = parser.parse_args()\n    args.out.parent.mkdir(parents=True, exist_ok=True)\n    model = PPO.load(args.model, device="cpu")\n    model.policy.eval()\n    wrapper = ExportPolicy(model.policy, model.action_space.low, model.action_space.high).eval()\n    dummy = torch.zeros((1, *model.observation_space.shape), dtype=torch.float32)\n    with torch.no_grad():\n        torch.onnx.export(wrapper, dummy, str(args.out), input_names=["observation"], output_names=["action"],\n                          dynamic_axes={"observation": {0: "batch"}, "action": {0: "batch"}},\n                          opset_version=17, dynamo=False)\n    onnx.checker.check_model(onnx.load(str(args.out)))\n    runtime = ort.InferenceSession(str(args.out), providers=["CPUExecutionProvider"])\n    # Cover small and extreme observations, so the output-clamping contract is checked too.\n    rng = np.random.default_rng(20261116)\n    observations = np.concatenate([np.zeros((1,4)), rng.normal(0, 0.2, (50,4)), rng.normal(0, 20, (50,4))]).astype(np.float32)\n    sb3, _ = model.predict(observations, deterministic=True)\n    exported = runtime.run(None, {"observation": observations})[0]\n    max_error = float(np.max(np.abs(sb3-exported)))\n    if not np.allclose(sb3, exported, rtol=1e-5, atol=1e-5):\n        raise RuntimeError(f"ONNX and SB3 differ, max error={max_error}")\n    evidence = {"samples": len(observations), "max_absolute_error": max_error,\n                "clip_inside_model": True, "observation_dtype": "float32",\n                "note": "simulation policy only; hardware deployment not tested"}\n    args.out.with_suffix(".verification.json").write_text(json.dumps(evidence, indent=2) + "\\n",encoding="utf-8")\n    print(json.dumps(evidence, indent=2))\n    print(f"Saved {args.out}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')


## 2. 训练并评估
先测随机基线，再训练连续动作 PPO；保存并重新加载后，用不同于训练种子的 20 个初始条件评估。训练需要一段时间，结果可能波动。


In [ ]:
!python train_ppo.py --steps 60000 --episodes 20


## 3. 看结果
平均步数只是这个仿真环境中的指标；1000 步是时间上限，不代表实机成功。


In [ ]:
import json
from pathlib import Path
report = json.loads(Path('runs/pendulum/evaluation.json').read_text(encoding='utf-8'))
print('随机策略平均步数:', report['random_baseline']['mean_steps'])
print('训练后平均步数:', report['trained_policy']['mean_steps'])
print('达到时间上限的回合:', report['trained_policy']['episodes_reaching_time_limit'], '/', len(report['evaluation_seeds']))
print('每回合训练后步数:', report['trained_policy']['lengths'])


## 4. 导出并验证策略
导出包含动作限幅的 ONNX；用 101 组输入与 SB3 比较，任何超出容差的差异会报错。ONNX 本身不是机器人固件，也不能直接驱动任意机器人。


In [ ]:
!python export_onnx.py


## 5. 打包自己的结果
在 Colab 中运行下一格可下载压缩包；其它 Jupyter 环境可直接在 `runs/pendulum` 找到结果。


In [ ]:
import shutil
shutil.make_archive('我的PPO仿真结果', 'zip', 'runs/pendulum')
try:
    from google.colab import files
    files.download('我的PPO仿真结果.zip')
except ImportError:
    print('已生成 我的PPO仿真结果.zip')
